# Module 7-P — PhysioNet EEGMMIDB Subject-Independent Motor-Imagery Classification

This notebook adapts the original Module 7 design to the PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB).

**Primary task:** 4-class motor imagery — Left Hand, Right Hand, Both Hands, Both Feet.

**Evaluation:** held-out-subject calibration + untouched held-out test runs. The final test labels are never used for training, temperature selection, fusion-weight selection, or model selection.

Pipeline: EDF loading → run-aware epoching → 4–38 Hz filtering → 128 Hz resampling → motor-channel selection → subject-wise Euclidean Alignment → FBCSP/shrinkage-LDA branch + multi-scale EEG CNN branch → held-out calibration fine-tuning → temperature scaling → confidence-aware fusion.

FBGAN is included as an optional extension but disabled by default because training one GAN per fold is expensive and may hurt stability on a Mac.


In [1]:
# CELL 1 — Install / verify dependencies
import sys, subprocess, pkgutil
REQ = {"mne":"mne","numpy":"numpy","scipy":"scipy","sklearn":"scikit-learn","torch":"torch","matplotlib":"matplotlib","pandas":"pandas"}
missing = [p for imp,p in REQ.items() if pkgutil.find_loader(imp) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
print("Missing installed:", missing)


Missing installed: []


In [2]:
# CELL 2 — Imports / device / reproducibility
import os, re, gc, time, copy, json, math, pickle, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.linalg import eigh
import mne
mne.set_log_level("ERROR")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

SEED=42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): device=torch.device("cuda")
elif hasattr(torch.backends,"mps") and torch.backends.mps.is_available(): device=torch.device("mps")
else: device=torch.device("cpu")
print("PyTorch:",torch.__version__,"Device:",device)


PyTorch: 2.10.0 Device: mps


In [62]:

# CELL 3 — FINAL PHYSIONET CONFIGURATION
# ============================================================

from pathlib import Path

# ============================================================
# DATA PATH
# ============================================================

DATA_ROOT = Path(
    "/Users/ashokvarmabevara/project2/eegmmidb"
)

if not DATA_ROOT.exists():

    # Fallback for relative notebook execution
    alt = Path(
        "./Users/ashokvarmabevara/project2/eegmmidb"
    )

    if alt.exists():
        DATA_ROOT = alt

    else:
        raise FileNotFoundError(
            "\nEEGMMIDB dataset not found.\n"
            "Expected:\n"
            "/Users/ashokvarmabevara/project2/eegmmidb\n"
        )

print(
    "Dataset:",
    DATA_ROOT
)


# ============================================================
# TASK
# ============================================================

TARGET_RUNS = [
    4,
    6,
    8,
    10,
    12,
    14
]

CALIBRATION_RUNS = [
    4,
    6,
    8
]

TEST_RUNS = [
    10,
    12,
    14
]

N_CLASSES = 4

CLASS_NAMES = [
    "Left Hand",
    "Right Hand",
    "Both Hands",
    "Both Feet"
]


# ============================================================
# SIGNAL
# ============================================================

ORIG_FS = 160
TARGET_FS = 128

BP_LO = 4.0
BP_HI = 38.0

NOTCH_HZ = 60.0


# ============================================================
# EPOCH WINDOWS
# ============================================================

# Main motor-imagery window
EPOCH_TMIN = 1.0
EPOCH_TMAX = 4.0

# Second overlapping window
USE_SECOND_WINDOW = True

SECOND_TMIN = 0.5
SECOND_TMAX = 3.5


# ============================================================
# EEG CHANNELS
# ============================================================

PREFERRED_MOTOR = [

    "FC1",
    "FC2",
    "FC3",
    "FC4",

    "C1",
    "C2",
    "C3",
    "Cz",
    "C4",
    "C5",
    "C6",

    "CP1",
    "CP2",
    "CP3",
    "CP4",
    "CP5",
    "CP6",

    "P1",
    "P2",
    "P3",
    "P4",
    "P5",
    "P6",
    "Pz",

    "F1",
    "F2",
    "F3",
    "F4",
    "Fz",

    "FC5",
    "FC6",
    "CPz"
]

N_CHANNELS_TARGET = 32


# ============================================================
# TRAINING
# ============================================================

BATCH_SIZE = 128

# Stronger than the previous 40-epoch version.
PRETRAIN_EPOCHS = 50

CALIB_EPOCHS = 8

PRETRAIN_LR = 5e-4

CALIB_LR = 2e-5

WEIGHT_DECAY = 1e-4


# ============================================================
# CENTER LOSS
# ============================================================

CENTER_LOSS_WEIGHT = 0.025

CENTER_LOSS_LR = 0.5

CENTER_WARMUP_EPOCHS = 10


# ============================================================
# AUGMENTATION
# ============================================================

AUG_PROB = 0.80

AMP_MIN = 0.90
AMP_MAX = 1.10

NOISE_STD = 0.010

MAX_TIME_SHIFT = 10


# ============================================================
# SHALLOW EEG MODEL
# ============================================================

TEMPORAL_FILTERS = 40

DROPOUT = 0.45


# ============================================================
# FBCSP
# ============================================================

CSP_BANDS = [

    (6, 10),
    (8, 12),
    (10, 14),
    (12, 18),
    (16, 24),
    (20, 30),
    (24, 36)
]

CSP_COMPONENTS = 6


# ============================================================
# FUSION
# ============================================================

# More trust in the neural branch because it learns
# cross-subject representations.
FIXED_FUSION_ALPHA = 0.70

# Alpha means:
# 0.70 * CNN
# 0.30 * CSP


# ============================================================
# TEMPERATURE
# ============================================================

TEMPERATURES = np.arange(
    0.70,
    2.51,
    0.05
)


# ============================================================
# GAN
# ============================================================

# OFF for the final deadline run.
USE_FBGAN = False


# ============================================================
# DEMO / LOSO
# ============================================================

RUN_MODE = "DEMO"

DEMO_SUBJECTS = 5


# ============================================================
# OUTPUT
# ============================================================

SAVE_DIR = (
    DATA_ROOT /
    "module7_final_outputs"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )


# ============================================================
# CACHE
# ============================================================

CACHE_FILE = (
    DATA_ROOT /
    "module7_physionet_cache_v2_multwindow.npz"
)


print("\n" + "=" * 65)

print(
    "FINAL CONFIGURATION"
)

print("=" * 65)

print(
    "Dataset       :",
    DATA_ROOT
)

print(
    "Device        :",
    device
)

print(
    "Runs          :",
    TARGET_RUNS
)

print(
    "Calibration   :",
    CALIBRATION_RUNS
)

print(
    "Test          :",
    TEST_RUNS
)

print(
    "Epoch window  :",
    EPOCH_TMIN,
    "→",
    EPOCH_TMAX
)

print(
    "Second window :",
    USE_SECOND_WINDOW
)

print(
    "Channels      :",
    N_CHANNELS_TARGET
)

print(
    "Pretrain      :",
    PRETRAIN_EPOCHS
)

print(
    "Calibration   :",
    CALIB_EPOCHS
)

print(
    "CSP components:",
    CSP_COMPONENTS
)

print(
    "Fusion alpha  :",
    FIXED_FUSION_ALPHA
)

print(
    "FBGAN         :",
    USE_FBGAN
)

print("=" * 65)



Dataset: /Users/ashokvarmabevara/project2/eegmmidb

FINAL CONFIGURATION
Dataset       : /Users/ashokvarmabevara/project2/eegmmidb
Device        : mps
Runs          : [4, 6, 8, 10, 12, 14]
Calibration   : [4, 6, 8]
Test          : [10, 12, 14]
Epoch window  : 1.0 → 4.0
Second window : True
Channels      : 32
Pretrain      : 50
Calibration   : 8
CSP components: 6
Fusion alpha  : 0.7
FBGAN         : False


In [63]:
# CELL 4 — Discover target EDF files
if not DATA_ROOT.exists(): raise FileNotFoundError(f"Dataset folder not found: {DATA_ROOT}")
edf_files=sorted(DATA_ROOT.rglob("S[0-9][0-9][0-9]R[0-9][0-9].edf"))
def parse_file_id(path):
    m=re.search(r"S(\d{3})R(\d{2})\.edf$",path.name,re.I)
    return (int(m.group(1)),int(m.group(2))) if m else (None,None)
records=[]
for f in edf_files:
    s,r=parse_file_id(f)
    if s is not None and r in TARGET_RUNS: records.append((s,r,f))
subjects_all=sorted({s for s,_,_ in records})
print(f"Target EDFs: {len(records)} | subjects: {len(subjects_all)}")
print("First:",subjects_all[:10],"Last:",subjects_all[-10:])
if len(subjects_all)<100: print("WARNING: expected roughly 109 subjects; inspect DATA_ROOT if fewer were found.")


Target EDFs: 654 | subjects: 109
First: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] Last: [100, 101, 102, 103, 104, 105, 106, 107, 108, 109]


In [64]:
# CELL 5 — Run-aware class mapping
# Runs 4/8/12: T1=left hand imagery, T2=right hand imagery
# Runs 6/10/14: T1=both hands imagery, T2=both feet imagery

def run_event_to_class(run, desc):
    d=str(desc).strip().upper()
    if d=="T0": return None
    if run in (4,8,12):
        return 0 if d=="T1" else (1 if d=="T2" else None)
    if run in (6,10,14):
        return 2 if d=="T1" else (3 if d=="T2" else None)
    return None

def choose_channels(ch_names, desired_n=32):
    lower={c.lower():c for c in ch_names}; chosen=[]
    for name in PREFERRED_MOTOR:
        if name.lower() in lower: chosen.append(lower[name.lower()])
    chosen=list(dict.fromkeys(chosen))
    if len(chosen)<desired_n:
        for c in ch_names:
            u=c.upper()
            if any(k in u for k in ["FC","C","CP","P"]) and c not in chosen: chosen.append(c)
            if len(chosen)>=desired_n: break
    if len(chosen)<desired_n:
        for c in ch_names:
            if c not in chosen: chosen.append(c)
            if len(chosen)>=desired_n: break
    return chosen[:desired_n]
print("Class mapping ready.")


Class mapping ready.


In [65]:
# CELL 6 — EDF loading + preprocessing + TWO-WINDOW epoch extraction
# ============================================================

def butter_bandpass_filter(x, lo, hi, fs, order=4):
    nyq = fs / 2.0
    hi = min(hi, nyq - 1.0)

    b, a = signal.butter(
        order,
        [lo / nyq, hi / nyq],
        btype="band"
    )

    return signal.filtfilt(
        b, a, x, axis=-1
    ).astype(np.float32)


def notch_filter(x, fs, f0=60.0, q=30.0):
    if f0 >= fs / 2:
        return x.astype(np.float32)

    b, a = signal.iirnotch(
        f0,
        q,
        fs
    )

    return signal.filtfilt(
        b, a, x, axis=-1
    ).astype(np.float32)


def robust_trial_standardize(x, eps=1e-6):
    """
    Per-channel robust standardization using median/MAD.
    """
    x = x - np.median(
        x,
        axis=-1,
        keepdims=True
    )

    scale = (
        np.median(
            np.abs(x),
            axis=-1,
            keepdims=True
        ) * 1.4826
    )

    return (
        x / (scale + eps)
    ).astype(np.float32)


def resample_trial(x, old_fs=160, new_fs=128):
    if int(old_fs) == int(new_fs):
        return x.astype(np.float32)

    n_new = int(
        round(
            x.shape[-1] *
            float(new_fs) /
            float(old_fs)
        )
    )

    return signal.resample(
        x,
        n_new,
        axis=-1
    ).astype(np.float32)


def run_event_to_class(run, desc):
    """
    PhysioNet EEGMMIDB motor imagery mapping.

    Runs 4,8,12:
        T1 = Left hand
        T2 = Right hand

    Runs 6,10,14:
        T1 = Both hands
        T2 = Both feet

    T0 = rest -> ignored
    """
    d = str(desc).strip().upper()

    if d == "T0":
        return None

    if run in (4, 8, 12):
        if d == "T1":
            return 0
        if d == "T2":
            return 1

    if run in (6, 10, 14):
        if d == "T1":
            return 2
        if d == "T2":
            return 3

    return None


def choose_channels(ch_names, desired_n=32):

    lower_map = {
        str(c).lower(): c
        for c in ch_names
    }

    selected = []

    for wanted in PREFERRED_MOTOR:
        key = wanted.lower()

        if key in lower_map:
            selected.append(
                lower_map[key]
            )

    selected = list(
        dict.fromkeys(selected)
    )

    # Add any remaining useful motor-area electrodes.
    if len(selected) < desired_n:

        extra = []

        for ch in ch_names:

            cu = str(ch).upper()

            if any(
                token in cu
                for token in [
                    "FC",
                    "C",
                    "CP",
                    "P"
                ]
            ):
                if ch not in selected:
                    extra.append(ch)

        selected.extend(extra)

    # Absolute fallback.
    if len(selected) < desired_n:

        for ch in ch_names:

            if ch not in selected:
                selected.append(ch)

            if len(selected) >= desired_n:
                break

    return selected[:desired_n]


def preprocess_trial(trial, fs):
    """
    trial shape = (channels, time)
    """
    trial = notch_filter(
        trial,
        fs,
        NOTCH_HZ
    )

    trial = butter_bandpass_filter(
        trial,
        BP_LO,
        BP_HI,
        fs
    )

    trial = resample_trial(
        trial,
        fs,
        TARGET_FS
    )

    trial = robust_trial_standardize(
        trial
    )

    return trial.astype(np.float32)


def extract_one_window(
    data,
    onset,
    fs,
    tmin,
    tmax
):
    start = int(
        round(
            (onset + tmin) *
            fs
        )
    )

    stop = int(
        round(
            (onset + tmax) *
            fs
        )
    )

    if (
        start < 0 or
        stop > data.shape[-1] or
        stop <= start
    ):
        return None

    trial = data[
        :,
        start:stop
    ]

    trial = preprocess_trial(
        trial,
        fs
    )

    expected_n = int(
        round(
            (tmax - tmin) *
            TARGET_FS
        )
    )

    if trial.shape[-1] != expected_n:
        return None

    return trial


def extract_run_epochs(
    edf_path,
    run
):

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    fs = float(
        raw.info["sfreq"]
    )

    ch_names = list(
        raw.ch_names
    )

    picks = choose_channels(
        ch_names,
        N_CHANNELS_TARGET
    )

    pick_idx = [
        ch_names.index(c)
        for c in picks
    ]

    data = raw.get_data(
        picks=pick_idx
    ).astype(np.float32)

    X_list = []
    y_list = []
    window_id_list = []

    annotations = raw.annotations

    for onset, desc in zip(
        annotations.onset,
        annotations.description
    ):

        cls = run_event_to_class(
            run,
            desc
        )

        if cls is None:
            continue

        # -----------------------------
        # WINDOW 1
        # -----------------------------
        trial1 = extract_one_window(
            data,
            onset,
            fs,
            EPOCH_TMIN,
            EPOCH_TMAX
        )

        if trial1 is not None:
            X_list.append(trial1)
            y_list.append(cls)
            window_id_list.append(1)

        # -----------------------------
        # WINDOW 2
        # -----------------------------
        if USE_SECOND_WINDOW:

            trial2 = extract_one_window(
                data,
                onset,
                fs,
                SECOND_TMIN,
                SECOND_TMAX
            )

            if trial2 is not None:
                X_list.append(trial2)
                y_list.append(cls)
                window_id_list.append(2)

    expected_t = int(
        round(
            (EPOCH_TMAX - EPOCH_TMIN)
            * TARGET_FS
        )
    )

    if not X_list:

        return (
            np.empty(
                (
                    0,
                    len(picks),
                    expected_t
                ),
                dtype=np.float32
            ),
            np.empty(
                0,
                dtype=np.int64
            ),
            np.empty(
                0,
                dtype=np.int64
            ),
            picks
        )

    return (
        np.stack(X_list).astype(np.float32),
        np.asarray(
            y_list,
            dtype=np.int64
        ),
        np.asarray(
            window_id_list,
            dtype=np.int64
        ),
        picks
    )


def load_subject_data(subject_id):

    sub_records = [
        (s, r, f)
        for s, r, f in records
        if s == subject_id
    ]

    Xs = []
    ys = []
    rs = []
    ws = []

    channel_names = None

    for _, run, f in sub_records:

        Xr, yr, wr, picks = extract_run_epochs(
            f,
            run
        )

        if len(Xr) == 0:
            continue

        Xs.append(Xr)
        ys.append(yr)

        rs.extend(
            [run] * len(yr)
        )

        ws.extend(
            wr.tolist()
        )

        if channel_names is None:
            channel_names = picks

    if not Xs:
        raise RuntimeError(
            f"No usable epochs for "
            f"subject {subject_id:03d}"
        )

    X = np.concatenate(
        Xs,
        axis=0
    ).astype(np.float32)

    y = np.concatenate(
        ys
    ).astype(np.int64)

    runs = np.asarray(
        rs,
        dtype=np.int64
    )

    windows = np.asarray(
        ws,
        dtype=np.int64
    )

    return (
        X,
        y,
        runs,
        windows,
        channel_names
    )


# ------------------------------------------------------------
# TEST SUBJECT 1
# ------------------------------------------------------------

X_test_subject, y_test_subject, r_test_subject, w_test_subject, ch_test = \
    load_subject_data(1)

print("S001")
print("X shape       :", X_test_subject.shape)
print("Class counts  :", np.bincount(
    y_test_subject,
    minlength=N_CLASSES
))
print("Run counts    :", {
    int(r): int(
        np.sum(r_test_subject == r)
    )
    for r in TARGET_RUNS
})
print("Window counts :", {
    int(w): int(
        np.sum(w_test_subject == w)
    )
    for w in np.unique(w_test_subject)
})
print("Channels      :", ch_test)

S001
X shape       : (180, 32, 384)
Class counts  : [46 44 42 48]
Run counts    : {4: 30, 6: 30, 8: 30, 10: 30, 12: 30, 14: 30}
Window counts : {1: 90, 2: 90}
Channels      : ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..', 'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.', 'Fp1.', 'Fpz.', 'Fp2.', 'Tp7.', 'Tp8.', 'P7..', 'P5..', 'P3..', 'P1..', 'Pz..', 'P2..']


In [66]:
# CELL 7 — Build / load v2 multi-window dataset
# ============================================================

def build_cache(
    subject_ids,
    cache_path
):

    all_X = []
    all_y = []
    all_s = []
    all_r = []
    all_w = []

    canonical_channels = None

    for i, sid in enumerate(
        subject_ids,
        1
    ):

        t0 = time.time()

        Xs, ys, rs, ws, ch = \
            load_subject_data(sid)

        if canonical_channels is None:

            canonical_channels = ch

        # Ensure channel count is identical.
        n = min(
            len(canonical_channels),
            Xs.shape[1]
        )

        Xs = Xs[:, :n]

        if canonical_channels is not None:
            canonical_channels = \
                canonical_channels[:n]

        all_X.append(Xs)

        all_y.append(ys)

        all_s.append(
            np.full(
                len(ys),
                sid,
                dtype=np.int16
            )
        )

        all_r.append(rs)

        all_w.append(ws)

        print(
            f"[{i:03d}/{len(subject_ids):03d}] "
            f"S{sid:03d}: "
            f"{len(ys):4d} epochs | "
            f"{time.time()-t0:5.1f}s"
        )

    X = np.concatenate(
        all_X,
        axis=0
    ).astype(np.float32)

    y = np.concatenate(
        all_y
    ).astype(np.int64)

    subjects_cached = np.concatenate(
        all_s
    ).astype(np.int16)

    runs_cached = np.concatenate(
        all_r
    ).astype(np.int16)

    windows_cached = np.concatenate(
        all_w
    ).astype(np.int16)

    np.savez_compressed(
        cache_path,
        X=X,
        y=y,
        subjects=subjects_cached,
        runs=runs_cached,
        windows=windows_cached,
        channels=np.asarray(
            canonical_channels,
            dtype=object
        ),
        fs=np.array(
            [TARGET_FS]
        )
    )

    return (
        X,
        y,
        subjects_cached,
        runs_cached,
        windows_cached,
        canonical_channels
    )


if CACHE_FILE.exists():

    print(
        "Loading:",
        CACHE_FILE
    )

    z = np.load(
        CACHE_FILE,
        allow_pickle=True
    )

    X_all = z["X"].astype(
        np.float32
    )

    y_all = z["y"].astype(
        np.int64
    )

    subject_arr = z[
        "subjects"
    ].astype(np.int16)

    run_arr = z[
        "runs"
    ].astype(np.int16)

    window_arr = z[
        "windows"
    ].astype(np.int16)

    channels = list(
        z["channels"]
    )

else:

    print(
        "Building new multi-window cache..."
    )

    X_all, y_all, subject_arr, run_arr, window_arr, channels = \
        build_cache(
            subjects_all,
            CACHE_FILE
        )


print("\nDataset")
print("---------------------------")
print("X             :", X_all.shape)
print("Subjects      :", len(np.unique(subject_arr)))
print("Class counts  :", np.bincount(
    y_all,
    minlength=N_CLASSES
))
print("Windows       :", np.unique(
    window_arr,
    return_counts=True
))

Loading: /Users/ashokvarmabevara/project2/eegmmidb/module7_physionet_cache_v2_multwindow.npz

Dataset
---------------------------
X             : (19675, 32, 384)
Subjects      : 109
Class counts  : [4959 4876 4930 4910]
Windows       : (array([1, 2], dtype=int16), array([9837, 9838]))


In [67]:
# CELL 8 — Euclidean Alignment
# ============================================================

def matrix_inv_sqrt(
    C,
    eps=1e-5
):

    vals, vecs = np.linalg.eigh(
        C.astype(np.float64)
    )

    vals = np.clip(
        vals,
        eps,
        None
    )

    return (
        vecs *
        (1.0 / np.sqrt(vals))
    ) @ vecs.T


def compute_ea_reference(
    X,
    max_trials=1600
):
    """
    Estimate reference covariance using unlabeled trials.

    X shape:
        (N, C, T)
    """

    if len(X) == 0:
        raise ValueError(
            "Cannot compute EA with zero trials."
        )

    n = len(X)

    if n > max_trials:

        idx = np.linspace(
            0,
            n - 1,
            max_trials,
            dtype=int
        )

    else:

        idx = np.arange(n)

    covs = []

    for i in idx:

        Xi = X[i].astype(
            np.float64
        )

        Xi = (
            Xi -
            Xi.mean(
                axis=1,
                keepdims=True
            )
        )

        C = (
            Xi @ Xi.T
        ) / max(
            Xi.shape[-1] - 1,
            1
        )

        C /= (
            np.trace(C)
            + 1e-8
        )

        covs.append(C)

    R = np.mean(
        covs,
        axis=0
    )

    R += (
        1e-3 *
        np.eye(R.shape[0])
    )

    return matrix_inv_sqrt(
        R
    ).astype(np.float32)


def apply_ea(
    X,
    W
):

    return np.einsum(
        "ij,njt->nit",
        W,
        X,
        optimize=True
    ).astype(np.float32)


def euclidean_alignment_subject(
    X,
    max_trials=1600
):

    W = compute_ea_reference(
        X,
        max_trials
    )

    return apply_ea(
        X,
        W
    )


def align_training_population(
    X,
    subjects
):
    """
    Training subjects:
    each subject gets its OWN EA matrix.

    No test subject enters this operation.
    """

    XA = np.empty_like(X)

    unique_subjects = np.unique(
        subjects
    )

    for sid in unique_subjects:

        m = subjects == sid

        W = compute_ea_reference(
            X[m]
        )

        XA[m] = apply_ea(
            X[m],
            W
        )

    return XA


def align_heldout_subject(
    X_subject,
    run_subject
):
    """
    Critical leakage fix.

    1. Estimate EA only from CALIBRATION_RUNS.
    2. Apply that exact matrix to calibration.
    3. Apply the SAME matrix to test.

    No test labels are used.
    """

    cal_mask = np.isin(
        run_subject,
        CALIBRATION_RUNS
    )

    test_mask = np.isin(
        run_subject,
        TEST_RUNS
    )

    X_cal_raw = X_subject[
        cal_mask
    ]

    X_test_raw = X_subject[
        test_mask
    ]

    if len(X_cal_raw) == 0:
        raise RuntimeError(
            "Held-out subject has "
            "no calibration data."
        )

    # EA estimated ONLY from calibration distribution.
    W = compute_ea_reference(
        X_cal_raw
    )

    X_cal = apply_ea(
        X_cal_raw,
        W
    )

    X_test = apply_ea(
        X_test_raw,
        W
    )

    return (
        X_cal,
        X_test,
        W
    )


print(
    "EA functions ready."
)

EA functions ready.


In [68]:

# CELL 9 — FINAL FIXED FBCSP + SHRINKAGE LDA
# ============================================================
#
# FIX:
#   Removed invalid references to y_cal and y_test inside
#   fit_fbcsp_lda().
#
#   Accuracy is calculated in run_subject_fold(), where the
#   correct calibration/test labels are available.
#
# ============================================================

from scipy.linalg import eigh
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


# ============================================================
# 1. BANDPASS FILTER
# ============================================================

def csp_bandpass(
    X,
    low,
    high,
    fs=TARGET_FS,
    order=4
):

    nyq = fs / 2.0

    high = min(
        float(high),
        nyq - 1.0
    )

    low = max(
        float(low),
        0.5
    )

    if low >= high:
        raise ValueError(
            f"Invalid CSP band: {low}-{high} Hz"
        )

    b, a = signal.butter(
        order,
        [
            low / nyq,
            high / nyq
        ],
        btype="band"
    )

    return signal.filtfilt(
        b,
        a,
        X,
        axis=-1
    ).astype(
        np.float32
    )


# ============================================================
# 2. SINGLE-TRIAL COVARIANCE
# ============================================================

def trial_covariance(
    trial,
    eps=1e-8
):

    trial = trial.astype(
        np.float64
    )

    # Remove channel-wise mean.
    trial = (
        trial
        - trial.mean(
            axis=1,
            keepdims=True
        )
    )

    C = (
        trial @ trial.T
    ) / max(
        trial.shape[-1] - 1,
        1
    )

    # Trace normalization.
    C /= (
        np.trace(C)
        + eps
    )

    return C


# ============================================================
# 3. MEAN COVARIANCE
# ============================================================

def mean_covariance(
    X
):

    if len(X) == 0:
        raise ValueError(
            "Cannot compute covariance "
            "from zero EEG trials."
        )

    covariances = []

    for trial in X:

        covariances.append(
            trial_covariance(
                trial
            )
        )

    return np.mean(
        covariances,
        axis=0
    )


# ============================================================
# 4. BINARY CSP
# ============================================================

def fit_binary_csp(
    X,
    y,
    target_class,
    n_components=CSP_COMPONENTS,
    reg=1e-3
):

    target_mask = (
        y == target_class
    )

    rest_mask = (
        y != target_class
    )

    X_target = X[
        target_mask
    ]

    X_rest = X[
        rest_mask
    ]

    if len(X_target) < 2:

        raise ValueError(
            f"Too few trials for class "
            f"{target_class}"
        )

    if len(X_rest) < 2:

        raise ValueError(
            f"Too few rest trials for class "
            f"{target_class}"
        )

    R_target = mean_covariance(
        X_target
    )

    R_rest = mean_covariance(
        X_rest
    )

    n_channels = \
        R_target.shape[0]

    regularizer = (
        reg *
        np.eye(
            n_channels
        )
    )

    A = (
        R_target +
        regularizer
    )

    B = (
        R_target +
        R_rest +
        regularizer
    )

    eigenvalues, eigenvectors = \
        eigh(
            A,
            B
        )

    # Highest eigenvalues first.
    order = np.argsort(
        eigenvalues
    )[::-1]

    eigenvectors = \
        eigenvectors[
            :,
            order
        ]

    n_components = int(
        min(
            n_components,
            n_channels
        )
    )

    # Take filters from both ends
    # of the eigenvalue spectrum.
    n_low = (
        n_components // 2
    )

    n_high = (
        n_components -
        n_low
    )

    low_idx = np.arange(
        n_channels -
        n_low,
        n_channels
    )

    high_idx = np.arange(
        0,
        n_high
    )

    selected = np.concatenate(
        [
            high_idx,
            low_idx
        ]
    )

    W = eigenvectors[
        :,
        selected
    ]

    return W.astype(
        np.float32
    )


# ============================================================
# 5. CSP LOG-VARIANCE FEATURES
# ============================================================

def csp_logvar_features(
    X,
    W,
    eps=1e-8
):

    # --------------------------------------------------------
    # Spatial filtering
    #
    # W shape:
    #   C × K
    #
    # output:
    #   N × K × T
    # --------------------------------------------------------

    projected = np.einsum(
        "kc,nct->nkt",
        W.T,
        X,
        optimize=True
    )

    # --------------------------------------------------------
    # Temporal variance
    # --------------------------------------------------------

    variance = np.var(
        projected,
        axis=-1
    )

    # --------------------------------------------------------
    # Normalize total CSP energy
    # --------------------------------------------------------

    variance /= (
        variance.sum(
            axis=1,
            keepdims=True
        )
        + eps
    )

    # --------------------------------------------------------
    # Log variance
    # --------------------------------------------------------

    features = np.log(
        variance + eps
    )

    return features.astype(
        np.float32
    )


# ============================================================
# 6. COMPLETE FBCSP + LDA
# ============================================================

def fit_fbcsp_lda(
    X_train,
    y_train,
    X_cal,
    X_test,
    bands=None,
    n_components=None
):

    if bands is None:

        bands = CSP_BANDS

    if n_components is None:

        n_components = CSP_COMPONENTS

    X_train = np.asarray(
        X_train,
        dtype=np.float32
    )

    X_cal = np.asarray(
        X_cal,
        dtype=np.float32
    )

    X_test = np.asarray(
        X_test,
        dtype=np.float32
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64
    )

    classes = np.unique(
        y_train
    )

    # --------------------------------------------------------
    # Validate classes
    # --------------------------------------------------------

    if len(classes) != N_CLASSES:

        raise ValueError(
            "FBCSP training data must "
            f"contain {N_CLASSES} classes. "
            f"Found: {classes}"
        )

    print(
        f"    FBCSP bands: {bands}"
    )

    all_train_features = []
    all_cal_features = []
    all_test_features = []

    fitted_filters = []

    # ========================================================
    # FILTER BANK
    # ========================================================

    for band_index, (
        low,
        high
    ) in enumerate(
        bands,
        start=1
    ):

        print(
            f"      Band "
            f"{band_index}/{len(bands)}: "
            f"{low}-{high} Hz",
            end=" ..."
        )

        # ----------------------------------------------------
        # Band-pass each dataset
        # ----------------------------------------------------

        Xtr_band = csp_bandpass(
            X_train,
            low,
            high
        )

        Xcal_band = csp_bandpass(
            X_cal,
            low,
            high
        )

        Xtest_band = csp_bandpass(
            X_test,
            low,
            high
        )

        band_filters = []

        train_features_parts = []
        cal_features_parts = []
        test_features_parts = []

        # ----------------------------------------------------
        # One-vs-rest CSP
        # ----------------------------------------------------

        for cls in classes:

            W = fit_binary_csp(
                Xtr_band,
                y_train,
                target_class=int(cls),
                n_components=n_components,
                reg=1e-3
            )

            band_filters.append(
                W
            )

            # TRAIN
            train_features_parts.append(
                csp_logvar_features(
                    Xtr_band,
                    W
                )
            )

            # CALIBRATION
            cal_features_parts.append(
                csp_logvar_features(
                    Xcal_band,
                    W
                )
            )

            # TEST
            test_features_parts.append(
                csp_logvar_features(
                    Xtest_band,
                    W
                )
            )

        # ----------------------------------------------------
        # Concatenate class-specific CSP features
        # ----------------------------------------------------

        train_band_features = \
            np.concatenate(
                train_features_parts,
                axis=1
            )

        cal_band_features = \
            np.concatenate(
                cal_features_parts,
                axis=1
            )

        test_band_features = \
            np.concatenate(
                test_features_parts,
                axis=1
            )

        all_train_features.append(
            train_band_features
        )

        all_cal_features.append(
            cal_band_features
        )

        all_test_features.append(
            test_band_features
        )

        fitted_filters.append(
            band_filters
        )

        print(
            f"features="
            f"{train_band_features.shape[1]}"
        )

        del Xtr_band
        del Xcal_band
        del Xtest_band

    # ========================================================
    # FULL FILTER BANK FEATURES
    # ========================================================

    F_train = np.concatenate(
        all_train_features,
        axis=1
    )

    F_cal = np.concatenate(
        all_cal_features,
        axis=1
    )

    F_test = np.concatenate(
        all_test_features,
        axis=1
    )

    print(
        "\n    Raw FBCSP feature shapes:"
    )

    print(
        "      Train:",
        F_train.shape
    )

    print(
        "      Cal  :",
        F_cal.shape
    )

    print(
        "      Test :",
        F_test.shape
    )

    # ========================================================
    # FINITE FEATURE SELECTION
    # ========================================================

    valid_train = np.isfinite(
        F_train
    ).all(
        axis=0
    )

    valid_cal = np.isfinite(
        F_cal
    ).all(
        axis=0
    )

    valid_test = np.isfinite(
        F_test
    ).all(
        axis=0
    )

    valid_features = (
        valid_train
        & valid_cal
        & valid_test
    )

    F_train = F_train[
        :,
        valid_features
    ]

    F_cal = F_cal[
        :,
        valid_features
    ]

    F_test = F_test[
        :,
        valid_features
    ]

    # ========================================================
    # STANDARDIZATION
    #
    # FIT ONLY ON TRAINING DATA
    # ========================================================

    scaler = StandardScaler(
        with_mean=True,
        with_std=True
    )

    F_train_scaled = \
        scaler.fit_transform(
            F_train
        )

    F_cal_scaled = \
        scaler.transform(
            F_cal
        )

    F_test_scaled = \
        scaler.transform(
            F_test
        )

    # ========================================================
    # SHRINKAGE LDA
    # ========================================================

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto"
    )

    lda.fit(
        F_train_scaled,
        y_train
    )

    # ========================================================
    # PROBABILITIES
    # ========================================================

    P_cal = lda.predict_proba(
        F_cal_scaled
    )

    P_test = lda.predict_proba(
        F_test_scaled
    )

    pred_cal = P_cal.argmax(
        axis=1
    )

    pred_test = P_test.argmax(
        axis=1
    )

    print(
        "\n    FBCSP complete."
    )

    print(
        "      Final features:",
        F_train_scaled.shape[1]
    )

    print(
        "      Classes:",
        lda.classes_
    )

    # IMPORTANT:
    # Do NOT calculate accuracy here because this function
    # intentionally does not receive y_cal/y_test.

    return {

        "filters":
            fitted_filters,

        "scaler":
            scaler,

        "lda":
            lda,

        "train_features":
            F_train_scaled,

        "cal_features":
            F_cal_scaled,

        "test_features":
            F_test_scaled,

        "Pcal":
            P_cal,

        "Ptest":
            P_test,

        "pred_cal":
            pred_cal,

        "pred_test":
            pred_test
    }


print(
    "\n✅ CELL 9 FIXED."
)

print(
    "fit_fbcsp_lda() is now available."
)



✅ CELL 9 FIXED.
fit_fbcsp_lda() is now available.


In [69]:

# CELL 10 — UPDATED MULTI-SCALE EEG CNN + CENTER LOSS
# ============================================================
#
# Architecture
#
#                Input EEG
#                  B,C,T
#                    │
#          ┌─────────┼─────────┐
#          │         │         │
#       k=15       k=31      k=63
#          │         │         │
#          └─────────┼─────────┘
#                    │
#             Multi-scale fusion
#                    │
#       Depthwise spatial filtering
#                    │
#             Temporal mixing
#                    │
#        Dilated residual blocks
#          d=1 → d=2 → d=4
#                    │
#               SE attention
#                    │
#           Global average pool
#                    │
#                64-D feature
#                    │
#             ┌──────┴──────┐
#             │             │
#        Center Loss      Classifier
#                            │
#                           4 classes
#
# ============================================================

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation attention.
    Learns channel-wise feature importance.
    """

    def __init__(
        self,
        channels,
        reduction=8
    ):
        super().__init__()

        hidden = max(
            4,
            channels // reduction
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(
                channels,
                hidden,
                bias=True
            ),
            nn.GELU(),
            nn.Linear(
                hidden,
                channels,
                bias=True
            ),
            nn.Sigmoid()
        )

    def forward(self, x):

        batch_size, channels, _, _ = x.shape

        scale = self.pool(x).view(
            batch_size,
            channels
        )

        scale = self.fc(
            scale
        ).view(
            batch_size,
            channels,
            1,
            1
        )

        return x * scale


class ResidualTemporalBlock(nn.Module):
    """
    Depthwise-separable temporal residual block.

    Dilation allows the model to observe different
    temporal scales without a very large parameter count.
    """

    def __init__(
        self,
        channels,
        kernel_size=9,
        dilation=1,
        dropout=0.10
    ):
        super().__init__()

        padding = (
            (kernel_size - 1) // 2
        ) * dilation

        self.depthwise = nn.Conv2d(
            channels,
            channels,
            kernel_size=(1, kernel_size),
            padding=(0, padding),
            dilation=(1, dilation),
            groups=channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            channels,
            channels,
            kernel_size=(1, 1),
            bias=False
        )

        self.bn = nn.BatchNorm2d(
            channels
        )

        self.act = nn.ELU(
            inplace=True
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        residual = x

        out = self.depthwise(
            x
        )

        out = self.pointwise(
            out
        )

        out = self.bn(
            out
        )

        out = self.act(
            out
        )

        out = self.dropout(
            out
        )

        return residual + out


class MultiScaleEEGNet(nn.Module):
    """
    Stronger EEG-specific network for PhysioNet EEGMMIDB.

    Input:
        (batch, channels, time)

    Output:
        logits  -> (batch, 4)
        feature -> (batch, 64)
    """

    def __init__(
        self,
        n_ch,
        n_t,
        n_cls=4,
        dropout=0.35
    ):
        super().__init__()

        self.n_ch = n_ch
        self.n_t = n_t
        self.n_cls = n_cls

        # ====================================================
        # 1. MULTI-SCALE TEMPORAL FILTERING
        # ====================================================

        F1 = 16

        self.temporal_short = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=F1,
                kernel_size=(1, 15),
                padding=(0, 7),
                bias=False
            ),

            nn.BatchNorm2d(
                F1
            ),

            nn.ELU(
                inplace=True
            )
        )

        self.temporal_medium = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=F1,
                kernel_size=(1, 31),
                padding=(0, 15),
                bias=False
            ),

            nn.BatchNorm2d(
                F1
            ),

            nn.ELU(
                inplace=True
            )
        )

        self.temporal_long = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=F1,
                kernel_size=(1, 63),
                padding=(0, 31),
                bias=False
            ),

            nn.BatchNorm2d(
                F1
            ),

            nn.ELU(
                inplace=True
            )
        )

        # Three temporal branches
        # 16 + 16 + 16 = 48
        merged_channels = F1 * 3

        # ====================================================
        # 2. DEPTHWISE SPATIAL FILTERING
        # ====================================================

        D = 2

        self.spatial = nn.Sequential(

            nn.Conv2d(
                in_channels=merged_channels,
                out_channels=merged_channels * D,
                kernel_size=(n_ch, 1),
                groups=merged_channels,
                bias=False
            ),

            nn.BatchNorm2d(
                merged_channels * D
            ),

            nn.ELU(
                inplace=True
            ),

            nn.AvgPool2d(
                kernel_size=(1, 4)
            ),

            nn.Dropout(
                dropout
            )
        )

        spatial_channels = \
            merged_channels * D

        # 48 * 2 = 96

        # ====================================================
        # 3. TEMPORAL FEATURE MIXER
        # ====================================================

        F2 = 48

        self.temporal_mixer = nn.Sequential(

            nn.Conv2d(
                in_channels=spatial_channels,
                out_channels=F2,
                kernel_size=(1, 15),
                padding=(0, 7),
                bias=False
            ),

            nn.BatchNorm2d(
                F2
            ),

            nn.ELU(
                inplace=True
            ),

            nn.AvgPool2d(
                kernel_size=(1, 2)
            ),

            nn.Dropout(
                dropout
            )
        )

        # ====================================================
        # 4. DILATED TEMPORAL RESIDUAL BLOCKS
        # ====================================================

        self.residual_1 = ResidualTemporalBlock(
            channels=F2,
            kernel_size=9,
            dilation=1,
            dropout=0.10
        )

        self.residual_2 = ResidualTemporalBlock(
            channels=F2,
            kernel_size=9,
            dilation=2,
            dropout=0.10
        )

        self.residual_3 = ResidualTemporalBlock(
            channels=F2,
            kernel_size=9,
            dilation=4,
            dropout=0.10
        )

        # ====================================================
        # 5. SQUEEZE-AND-EXCITATION
        # ====================================================

        self.se = SEBlock(
            channels=F2,
            reduction=8
        )

        # ====================================================
        # 6. GLOBAL POOLING
        # ====================================================

        self.global_pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        # ====================================================
        # 7. 64-D SUBJECT-INVARIANT REPRESENTATION
        # ====================================================

        self.feature_layer = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                F2,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )

        self.feature_dim = 64

        # ====================================================
        # 8. CLASSIFICATION HEAD
        # ====================================================

        self.classifier = nn.Linear(
            self.feature_dim,
            n_cls
        )

    def forward(
        self,
        x
    ):
        """
        x:
            B,C,T
        """

        # ----------------------------------------------------
        # Convert to 2D convolution format
        # B,C,T -> B,1,C,T
        # ----------------------------------------------------

        x = x.unsqueeze(
            dim=1
        )

        # ----------------------------------------------------
        # Parallel temporal filters
        # ----------------------------------------------------

        x_short = self.temporal_short(
            x
        )

        x_medium = self.temporal_medium(
            x
        )

        x_long = self.temporal_long(
            x
        )

        # ----------------------------------------------------
        # Multi-scale feature fusion
        # ----------------------------------------------------

        x = torch.cat(
            [
                x_short,
                x_medium,
                x_long
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Spatial filtering across ALL EEG channels
        # ----------------------------------------------------

        x = self.spatial(
            x
        )

        # ----------------------------------------------------
        # Temporal mixing
        # ----------------------------------------------------

        x = self.temporal_mixer(
            x
        )

        # ----------------------------------------------------
        # Multi-resolution temporal modeling
        # ----------------------------------------------------

        x = self.residual_1(
            x
        )

        x = self.residual_2(
            x
        )

        x = self.residual_3(
            x
        )

        # ----------------------------------------------------
        # Channel attention
        # ----------------------------------------------------

        x = self.se(
            x
        )

        # ----------------------------------------------------
        # Global average pooling
        # ----------------------------------------------------

        x = self.global_pool(
            x
        )

        # ----------------------------------------------------
        # 64-D embedding
        # ----------------------------------------------------

        features = self.feature_layer(
            x
        )

        # ----------------------------------------------------
        # Final classifier
        # ----------------------------------------------------

        logits = self.classifier(
            features
        )

        return logits, features


# ============================================================
# CENTER LOSS
# ============================================================

class CenterLoss(nn.Module):
    """
    Center loss encourages samples from the same class
    to occupy compact regions in the learned feature space.

    This is especially useful for subject-independent EEG.
    """

    def __init__(
        self,
        n_classes,
        feature_dim,
        center_lr=0.5
    ):
        super().__init__()

        self.n_classes = n_classes
        self.feature_dim = feature_dim
        self.center_lr = center_lr

        self.register_buffer(
            "centers",
            torch.zeros(
                n_classes,
                feature_dim
            )
        )

    @torch.no_grad()
    def initialize_from_data(
        self,
        model,
        X,
        y,
        batch_size=256
    ):
        """
        Initialize each class center from the
        population training embeddings.
        """

        model.eval()

        dataset = TensorDataset(
            torch.from_numpy(
                X
            ).float(),
            torch.from_numpy(
                y
            ).long()
        )

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False
        )

        feature_parts = []
        label_parts = []

        for xb, yb in loader:

            xb = xb.to(
                device
            )

            _, features = model(
                xb
            )

            feature_parts.append(
                features.detach().cpu()
            )

            label_parts.append(
                yb
            )

        all_features = torch.cat(
            feature_parts,
            dim=0
        )

        all_labels = torch.cat(
            label_parts,
            dim=0
        )

        # ----------------------------------------------
        # Initialize one center per class
        # ----------------------------------------------

        for cls in range(
            self.n_classes
        ):

            mask = (
                all_labels == cls
            )

            if mask.sum() > 0:

                self.centers[
                    cls
                ] = all_features[
                    mask
                ].mean(
                    dim=0
                ).to(
                    self.centers.device
                )

        model.train()

    @torch.no_grad()
    def update_centers(
        self,
        features,
        labels
    ):
        """
        Exponential-style center update.
        """

        for cls in range(
            self.n_classes
        ):

            mask = (
                labels == cls
            )

            if mask.sum() == 0:
                continue

            class_mean = \
                features[
                    mask
                ].detach().mean(
                    dim=0
                )

            delta = (
                self.centers[cls]
                - class_mean
            )

            self.centers[
                cls
            ] -= (
                self.center_lr *
                delta
            )

    def forward(
        self,
        features,
        labels
    ):
        """

        Center loss:
            || feature - class_center ||

        """

        class_centers = \
            self.centers[
                labels
            ]

        distance = torch.norm(
            features -
            class_centers,
            dim=1
        )

        return distance.mean()


# ============================================================
# MODEL FACTORY
# ============================================================

def make_model(
    n_ch,
    n_t
):

    model = MultiScaleEEGNet(
        n_ch=n_ch,
        n_t=n_t,
        n_cls=N_CLASSES,
        dropout=0.35
    ).to(
        device
    )

    return model


# ============================================================
# MODEL SHAPE TEST
# ============================================================

print(
    "\nTesting updated model..."
)

_test_model = make_model(
    X_all.shape[1],
    X_all.shape[2]
)

_test_model.eval()

with torch.no_grad():

    dummy = torch.zeros(
        2,
        X_all.shape[1],
        X_all.shape[2],
        device=device
    )

    test_logits, test_features = \
        _test_model(
            dummy
        )

print(
    "Input shape    :",
    dummy.shape
)

print(
    "Logits shape   :",
    test_logits.shape
)

print(
    "Feature shape  :",
    test_features.shape
)

print(
    "Feature dim    :",
    _test_model.feature_dim
)

print(
    "Parameters     :",
    f"{sum(p.numel() for p in _test_model.parameters()):,}"
)

assert test_logits.shape == (
    2,
    N_CLASSES
)

assert test_features.shape == (
    2,
    64
)

print(
    "\n✅ Updated Cell 10 model check passed."
)

del _test_model
del dummy
del test_logits
del test_features

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()




Testing updated model...
Input shape    : torch.Size([2, 32, 384])
Logits shape   : torch.Size([2, 4])
Feature shape  : torch.Size([2, 64])
Feature dim    : 64
Parameters     : 86,842

✅ Updated Cell 10 model check passed.


In [70]:

# CELL 11 — FINAL EEG AUGMENTATION + LOADERS
# ============================================================


def augment_batch(
    x
):

    if np.random.rand() > AUG_PROB:
        return x

    B, C, T = x.shape

    out = x.clone()

    # ========================================================
    # 1. Amplitude scaling
    # ========================================================

    scale = torch.empty(
        B,
        1,
        1,
        device=x.device
    ).uniform_(
        AMP_MIN,
        AMP_MAX
    )

    out = out * scale

    # ========================================================
    # 2. Small Gaussian noise
    # ========================================================

    if np.random.rand() < 0.70:

        out = (
            out +
            NOISE_STD *
            torch.randn_like(out)
        )

    # ========================================================
    # 3. Temporal shift
    # ========================================================

    if np.random.rand() < 0.70:

        shifts = torch.randint(
            -MAX_TIME_SHIFT,
            MAX_TIME_SHIFT + 1,
            (B,),
            device=x.device
        )

        shifted = []

        for i, shift in enumerate(
            shifts.tolist()
        ):

            shifted.append(
                torch.roll(
                    out[i],
                    int(shift),
                    dims=-1
                )
            )

        out = torch.stack(
            shifted,
            dim=0
        )

    # ========================================================
    # 4. Small temporal masking
    # ========================================================

    if np.random.rand() < 0.25:

        width = max(
            4,
            T // 24
        )

        for i in range(B):

            start = np.random.randint(
                0,
                max(
                    1,
                    T - width
                )
            )

            out[
                i,
                :,
                start:start + width
            ] *= 0.85

    # ========================================================
    # 5. Conservative channel dropout
    # ========================================================

    if np.random.rand() < 0.20:

        n_drop = max(
            1,
            C // 24
        )

        mask = torch.ones(
            B,
            C,
            1,
            device=x.device
        )

        for i in range(B):

            idx = torch.randperm(
                C,
                device=x.device
            )[:n_drop]

            mask[
                i,
                idx
            ] = 0.0

        out = out * mask

    return out


# ============================================================
# CLASS-BALANCED LOADER
# ============================================================

def weighted_loader(
    X,
    y,
    batch_size=BATCH_SIZE
):

    counts = np.bincount(
        y,
        minlength=N_CLASSES
    ).astype(
        np.float64
    )

    weights = np.array(
        [
            1.0 /
            max(
                counts[int(c)],
                1.0
            )
            for c in y
        ],
        dtype=np.float64
    )

    sampler = WeightedRandomSampler(
        torch.as_tensor(
            weights,
            dtype=torch.double
        ),
        num_samples=len(y),
        replacement=True
    )

    dataset = TensorDataset(
        torch.from_numpy(
            X
        ).float(),
        torch.from_numpy(
            y
        ).long()
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        drop_last=False
    )


print(
    "✅ Cell 11 ready."
)



✅ Cell 11 ready.


In [71]:

# CELL 12 — FINAL TRAINING / PREDICTION
# ============================================================


def softmax_np(
    logits
):

    logits = (
        logits -
        logits.max(
            axis=1,
            keepdims=True
        )
    )

    exp_logits = np.exp(
        logits
    )

    return (
        exp_logits /
        (
            exp_logits.sum(
                axis=1,
                keepdims=True
            )
            + 1e-12
        )
    )


# ============================================================
# PREDICTION
# ============================================================

def run_predictions(
    model,
    X,
    batch_size=256
):

    model.eval()

    dataset = TensorDataset(
        torch.from_numpy(
            X
        ).float()
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    logits_list = []

    with torch.no_grad():

        for (xb,) in loader:

            logits, _ = model(
                xb.to(device)
            )

            logits_list.append(
                logits.cpu().numpy()
            )

    logits = np.vstack(
        logits_list
    )

    probs = softmax_np(
        logits
    )

    return (
        logits,
        probs
    )


# ============================================================
# TEMPERATURE
# ============================================================

def fit_temperature(
    logits,
    y
):

    best_temperature = 1.0
    best_nll = np.inf

    y = np.asarray(
        y
    )

    for temperature in TEMPERATURES:

        probs = softmax_np(
            logits /
            float(temperature)
        )

        true_probs = np.clip(
            probs[
                np.arange(
                    len(y)
                ),
                y
            ],
            1e-12,
            1.0
        )

        nll = -np.mean(
            np.log(
                true_probs
            )
        )

        if nll < best_nll:

            best_nll = nll

            best_temperature = \
                float(temperature)

    return (
        best_temperature,
        best_nll
    )


# ============================================================
# POPULATION MODEL TRAINING
# ============================================================

def train_population_model(
    X_train,
    y_train,
    epochs=PRETRAIN_EPOCHS,
    verbose=True
):

    model = make_model(
        X_train.shape[1],
        X_train.shape[2]
    )

    center_loss = CenterLoss(
        n_classes=N_CLASSES,
        feature_dim=model.feature_dim,
        center_lr=CENTER_LOSS_LR
    ).to(
        device
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.05
    )

    loader = weighted_loader(
        X_train,
        y_train
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PRETRAIN_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = \
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
            eta_min=1e-5
        )

    # --------------------------------------------------------
    # Initialize center loss after warm-up
    # --------------------------------------------------------

    center_initialized = False

    best_state = None

    best_train_acc = -1.0

    history = {
        "loss": [],
        "ce": [],
        "center": [],
        "train_acc": []
    }

    for epoch in range(
        1,
        epochs + 1
    ):

        # ----------------------------------------------------
        # Center initialization
        # ----------------------------------------------------

        if (
            epoch ==
            CENTER_WARMUP_EPOCHS + 1
            and
            not center_initialized
        ):

            print(
                "    Initializing class centers..."
            )

            center_loss.initialize_from_data(
                model,
                X_train,
                y_train
            )

            center_initialized = True

        # ----------------------------------------------------
        # Training mode
        # ----------------------------------------------------

        model.train()

        total_loss = 0.0
        total_ce = 0.0
        total_center = 0.0

        correct = 0
        total = 0

        use_center = (
            epoch >
            CENTER_WARMUP_EPOCHS
        )

        # ----------------------------------------------------
        # Batches
        # ----------------------------------------------------

        for xb, yb in loader:

            xb = xb.to(
                device
            )

            yb = yb.to(
                device
            )

            # EEG augmentation
            xb = augment_batch(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, features = \
                model(
                    xb
                )

            loss_ce = criterion(
                logits,
                yb
            )

            if use_center:

                loss_center = \
                    center_loss(
                        features,
                        yb
                    )

                loss = (
                    loss_ce
                    +
                    CENTER_LOSS_WEIGHT
                    *
                    loss_center
                )

            else:

                loss_center = torch.tensor(
                    0.0,
                    device=device
                )

                loss = loss_ce

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            if use_center:

                center_loss.update_centers(
                    features,
                    yb
                )

            total_loss += float(
                loss.item()
            )

            total_ce += float(
                loss_ce.item()
            )

            total_center += float(
                loss_center.item()
            )

            correct += int(
                (
                    logits.argmax(
                        dim=1
                    )
                    ==
                    yb
                ).sum()
            )

            total += len(
                yb
            )

        scheduler.step()

        train_acc = (
            correct /
            max(
                total,
                1
            )
        )

        avg_loss = (
            total_loss /
            max(
                len(loader),
                1
            )
        )

        avg_ce = (
            total_ce /
            max(
                len(loader),
                1
            )
        )

        avg_center = (
            total_center /
            max(
                len(loader),
                1
            )
        )

        history[
            "loss"
        ].append(
            avg_loss
        )

        history[
            "ce"
        ].append(
            avg_ce
        )

        history[
            "center"
        ].append(
            avg_center
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        # ----------------------------------------------------
        # Save best training model
        # ----------------------------------------------------

        if train_acc > best_train_acc:

            best_train_acc = \
                train_acc

            best_state = \
                copy.deepcopy(
                    model.state_dict()
                )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if verbose and (
            epoch == 1
            or epoch % 5 == 0
            or epoch == epochs
        ):

            phase = (
                "CENTER"
                if use_center
                else "WARMUP"
            )

            print(
                f"    epoch "
                f"{epoch:02d}/{epochs} "
                f"[{phase}] "
                f"loss={avg_loss:.4f} "
                f"CE={avg_ce:.4f} "
                f"C={avg_center:.4f} "
                f"train={train_acc*100:.2f}%"
            )

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return (
        model,
        history
    )


# ============================================================
# CALIBRATION FINE-TUNING
# ============================================================

def finetune_calibration(
    model,
    X_cal,
    y_cal,
    epochs=CALIB_EPOCHS
):

    # --------------------------------------------------------
    # Freeze early feature extractor.
    #
    # Only the later representation and classifier adapt
    # to the held-out subject.
    # --------------------------------------------------------

    for name, parameter in \
        model.named_parameters():

        parameter.requires_grad = (
            name.startswith(
                "temporal2"
            )
            or
            name.startswith(
                "se"
            )
            or
            name.startswith(
                "feature_layer"
            )
            or
            name.startswith(
                "classifier"
            )
        )

    trainable_parameters = list(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        )
    )

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=CALIB_LR,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.05
    )

    batch_size = max(
        16,
        min(
            64,
            len(y_cal) // 4
        )
    )

    loader = weighted_loader(
        X_cal,
        y_cal,
        batch_size=batch_size
    )

    best_state = \
        copy.deepcopy(
            model.state_dict()
        )

    best_acc = -1.0

    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        # Keep BatchNorm statistics fixed
        for module in model.modules():

            if isinstance(
                module,
                nn.BatchNorm2d
            ):

                module.eval()

        correct = 0
        total = 0

        for xb, yb in loader:

            xb = xb.to(
                device
            )

            yb = yb.to(
                device
            )

            xb = augment_batch(
                xb
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, _ = model(
                xb
            )

            loss = criterion(
                logits,
                yb
            )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                0.5
            )

            optimizer.step()

            correct += int(
                (
                    logits.argmax(
                        dim=1
                    )
                    ==
                    yb
                ).sum()
            )

            total += len(
                yb
            )

        acc = (
            correct /
            max(
                total,
                1
            )
        )

        if acc > best_acc:

            best_acc = acc

            best_state = \
                copy.deepcopy(
                    model.state_dict()
                )

    model.load_state_dict(
        best_state
    )

    for parameter in \
        model.parameters():

        parameter.requires_grad = True

    return (
        model,
        best_acc
    )


print(
    "\n✅ CELL 12 READY."
)




✅ CELL 12 READY.


In [72]:

# CELL 13 — FINAL STABLE FUSION
# ============================================================


def confidence_fusion(
    P_csp,
    P_cnn,
    alpha=FIXED_FUSION_ALPHA
):
    """
    Weighted probability fusion.

    alpha = CNN weight.
    1-alpha = CSP weight.
    """

    P = (
        alpha * P_cnn
        +
        (1.0 - alpha) * P_csp
    )

    P = P / (
        P.sum(
            axis=1,
            keepdims=True
        )
        + 1e-12
    )

    return P


def choose_calibration_alpha(
    P_csp_cal,
    P_cnn_cal,
    y_cal
):

    candidates = [
        0.60,
        0.65,
        0.70,
        0.75,
        0.80
    ]

    best_alpha = \
        FIXED_FUSION_ALPHA

    best_acc = -1.0

    for alpha in candidates:

        P = confidence_fusion(
            P_csp_cal,
            P_cnn_cal,
            alpha
        )

        acc = accuracy_score(
            y_cal,
            P.argmax(
                axis=1
            )
        )

        if acc > best_acc:

            best_acc = acc

            best_alpha = alpha

    return (
        best_alpha,
        best_acc
    )


print(
    "✅ Stable fusion ready."
)

print(
    "Default CNN weight:",
    FIXED_FUSION_ALPHA
)


✅ Stable fusion ready.
Default CNN weight: 0.7


In [73]:
# CELL 14 — One held-out subject fold

def run_subject_fold(test_subject,verbose=False):
    mtrain=subject_arr!=test_subject; msub=subject_arr==test_subject
    Xtr_raw=X_all[mtrain]; ytr=y_all[mtrain]
    # Align every training subject independently using only that subject's signals.
    Xtr=np.empty_like(Xtr_raw)
    local_train_subjects=np.unique(subject_arr[mtrain])
    for sid in local_train_subjects:
        local=(subject_arr[mtrain]==sid); Xtr[local]=ea_subject(Xtr_raw[local])
    # Held-out subject alignment uses unlabeled held-out data only.
    Xsub= X_all[msub]; rsub=run_arr[msub]; Xsub_ea=ea_subject(Xsub)
    Xcal=Xsub_ea[np.isin(rsub,CALIBRATION_RUNS)]; ycal=y_all[msub][np.isin(rsub,CALIBRATION_RUNS)]
    Xte=Xsub_ea[np.isin(rsub,TEST_RUNS)]; yte=y_all[msub][np.isin(rsub,TEST_RUNS)]
    if len(np.unique(ycal))<4 or len(np.unique(yte))<4: raise RuntimeError(f"S{test_subject:03d} lacks all 4 classes in cal/test")

    csp=fit_fbcsp(Xtr,ytr,Xcal,Xte); Pc_cal,Pc_te=csp["Pcal"],csp["Pte"]
    model=train_population_model(Xtr,ytr,verbose=verbose)
    L0,_=run_predictions(model,Xcal); T0,_=fit_temperature(L0,ycal); P0_cal=softmax_np(L0/T0)
    model_ft,cal_train=finetune_calibration(copy.deepcopy(model),Xcal,ycal)
    Lcal,_=run_predictions(model_ft,Xcal); Lte,_=run_predictions(model_ft,Xte); Tft,_=fit_temperature(Lcal,ycal)
    Pn_cal=softmax_np(Lcal/Tft); Pn_te=softmax_np(Lte/Tft)
    alpha,acc_cal=choose_alpha(Pc_cal,Pn_cal,ycal); Pfu_te=confidence_fusion(Pc_te,Pn_te,alpha)
    acsp=accuracy_score(yte,Pc_te.argmax(1)); anet=accuracy_score(yte,Pn_te.argmax(1)); afu=accuracy_score(yte,Pfu_te.argmax(1))
    candidates={"FBCSP-LDA":(accuracy_score(ycal,Pc_cal.argmax(1)),Pc_te),"CNN":(accuracy_score(ycal,Pn_cal.argmax(1)),Pn_te),"FBCSP+CNN":(acc_cal,Pfu_te)}
    best_method=max(candidates,key=lambda k:candidates[k][0]); best_cal,bestP=candidates[best_method]; best_pred=bestP.argmax(1); best_te=accuracy_score(yte,best_pred)
    return {"subject":int(test_subject),"best_method":best_method,"best_cal_acc":best_cal,"best_test_acc":best_te,"csp_test_acc":acsp,"cnn_test_acc":anet,"fusion_test_acc":afu,"alpha":alpha,"T0":T0,"Tft":Tft,"cal_train_acc":cal_train,"y_test":yte,"pred_test":best_pred,"proba_test":bestP}


In [74]:
# QUICK DATASET CHECK

for sid in [1, 2, 3, 4, 5]:

    m = subject_arr == sid

    print(
        f"S{sid:03d}: "
        f"total={m.sum()} | "
        f"classes="
        f"{np.bincount(y_all[m], minlength=4)}"
    )

S001: total=180 | classes=[46 44 42 48]
S002: total=180 | classes=[46 44 48 42]
S003: total=180 | classes=[46 44 42 48]
S004: total=180 | classes=[46 44 44 46]
S005: total=180 | classes=[42 48 46 44]


In [75]:

# CELL 15 — FINAL ROBUST SUBJECT FOLD
# ============================================================
#
# THIS VERSION:
#   - uses calibration-only EA for held-out subject
#   - trains FBCSP on population subjects only
#   - trains ShallowEEGNet on population subjects only
#   - fine-tunes only with held-out calibration data
#   - never uses test labels for model fitting
#   - returns one stable dictionary structure
#
# ============================================================


def run_subject_fold(
    test_subject,
    verbose=False
):

    test_subject = int(
        test_subject
    )

    print(
        f"\nPreparing population EA "
        f"for held-out S{test_subject:03d}..."
    )

    # ========================================================
    # 1. POPULATION TRAINING SPLIT
    # ========================================================

    train_mask = (
        subject_arr != test_subject
    )

    heldout_mask = (
        subject_arr == test_subject
    )

    X_train_raw = X_all[
        train_mask
    ]

    y_train = y_all[
        train_mask
    ]

    train_subjects = subject_arr[
        train_mask
    ]

    X_subject_raw = X_all[
        heldout_mask
    ]

    y_subject = y_all[
        heldout_mask
    ]

    runs_subject = run_arr[
        heldout_mask
    ]

    # ========================================================
    # 2. POPULATION EA
    # ========================================================

    X_train = np.empty_like(
        X_train_raw
    )

    unique_train_subjects = \
        np.unique(
            train_subjects
        )

    for sid in unique_train_subjects:

        local_mask = (
            train_subjects == sid
        )

        X_sid = X_train_raw[
            local_mask
        ]

        W_sid = \
            compute_ea_reference(
                X_sid
            )

        X_train[
            local_mask
        ] = apply_ea(
            X_sid,
            W_sid
        )

    # ========================================================
    # 3. HELD-OUT CALIBRATION / TEST
    # ========================================================

    calibration_mask = np.isin(
        runs_subject,
        CALIBRATION_RUNS
    )

    test_mask = np.isin(
        runs_subject,
        TEST_RUNS
    )

    X_cal_raw = X_subject_raw[
        calibration_mask
    ]

    X_test_raw = X_subject_raw[
        test_mask
    ]

    y_cal = y_subject[
        calibration_mask
    ]

    y_test = y_subject[
        test_mask
    ]

    # ========================================================
    # 4. CALIBRATION-ONLY EA
    # ========================================================

    W_calibration = \
        compute_ea_reference(
            X_cal_raw
        )

    X_cal = apply_ea(
        X_cal_raw,
        W_calibration
    )

    X_test = apply_ea(
        X_test_raw,
        W_calibration
    )

    print(
        "Train:",
        X_train.shape,
        "Cal:",
        X_cal.shape,
        "Test:",
        X_test.shape
    )

    print(
        "Cal classes:",
        np.bincount(
            y_cal,
            minlength=N_CLASSES
        )
    )

    print(
        "Test classes:",
        np.bincount(
            y_test,
            minlength=N_CLASSES
        )
    )

    # ========================================================
    # 5. FBCSP
    # ========================================================

    print(
        "\nTraining FBCSP-LDA..."
    )

    csp = fit_fbcsp_lda(
        X_train,
        y_train,
        X_cal,
        X_test
    )

    P_csp_cal = csp[
        "Pcal"
    ]

    # Compatible with both possible naming conventions
    if "Ptest" in csp:

        P_csp_test = \
            csp["Ptest"]

    elif "Pte" in csp:

        P_csp_test = \
            csp["Pte"]

    else:

        raise KeyError(
            "FBCSP must return Ptest or Pte."
        )

    pred_csp_cal = \
        P_csp_cal.argmax(
            axis=1
        )

    pred_csp_test = \
        P_csp_test.argmax(
            axis=1
        )

    csp_cal_acc = accuracy_score(
        y_cal,
        pred_csp_cal
    )

    csp_test_acc = accuracy_score(
        y_test,
        pred_csp_test
    )

    print(
        f"    CSP calibration: "
        f"{csp_cal_acc*100:.2f}%"
    )

    print(
        f"    CSP test       : "
        f"{csp_test_acc*100:.2f}%"
    )

    # ========================================================
    # 6. POPULATION SHALLOW CNN
    # ========================================================

    print(
        "\nTraining ShallowEEGNet..."
    )

    model_population, history = \
        train_population_model(
            X_train,
            y_train,
            epochs=PRETRAIN_EPOCHS,
            verbose=verbose
        )

    # ========================================================
    # 7. POPULATION CNN PREDICTION
    # ========================================================

    logits_cal_population, _ = \
        run_predictions(
            model_population,
            X_cal
        )

    logits_test_population, _ = \
        run_predictions(
            model_population,
            X_test
        )

    population_T, _ = \
        fit_temperature(
            logits_cal_population,
            y_cal
        )

    P_population_cal = \
        softmax_np(
            logits_cal_population /
            population_T
        )

    P_population_test = \
        softmax_np(
            logits_test_population /
            population_T
        )

    population_cal_acc = \
        accuracy_score(
            y_cal,
            P_population_cal.argmax(
                axis=1
            )
        )

    population_test_acc = \
        accuracy_score(
            y_test,
            P_population_test.argmax(
                axis=1
            )
        )

    print(
        f"    Population CNN "
        f"cal={population_cal_acc*100:.2f}% "
        f"test={population_test_acc*100:.2f}%"
    )

    # ========================================================
    # 8. SUBJECT CALIBRATION FINE-TUNING
    # ========================================================

    print(
        "\nFine-tuning on calibration..."
    )

    model_finetuned = copy.deepcopy(
        model_population
    )

    model_finetuned, \
        calibration_train_acc = \
        finetune_calibration(
            model_finetuned,
            X_cal,
            y_cal,
            epochs=CALIB_EPOCHS
        )

    # ========================================================
    # 9. FINE-TUNED CNN
    # ========================================================

    logits_cal_ft, _ = \
        run_predictions(
            model_finetuned,
            X_cal
        )

    logits_test_ft, _ = \
        run_predictions(
            model_finetuned,
            X_test
        )

    fine_tuned_T, _ = \
        fit_temperature(
            logits_cal_ft,
            y_cal
        )

    P_cnn_cal = \
        softmax_np(
            logits_cal_ft /
            fine_tuned_T
        )

    P_cnn_test = \
        softmax_np(
            logits_test_ft /
            fine_tuned_T
        )

    pred_cnn_cal = \
        P_cnn_cal.argmax(
            axis=1
        )

    pred_cnn_test = \
        P_cnn_test.argmax(
            axis=1
        )

    cnn_cal_acc = \
        accuracy_score(
            y_cal,
            pred_cnn_cal
        )

    cnn_test_acc = \
        accuracy_score(
            y_test,
            pred_cnn_test
        )

    print(
        f"    Fine-tuned CNN "
        f"cal={cnn_cal_acc*100:.2f}% "
        f"test={cnn_test_acc*100:.2f}%"
    )

    # ========================================================
    # 10. FIXED FUSION
    # ========================================================

    P_fixed_cal = \
        confidence_fusion(
            P_csp_cal,
            P_cnn_cal,
            FIXED_FUSION_ALPHA
        )

    P_fixed_test = \
        confidence_fusion(
            P_csp_test,
            P_cnn_test,
            FIXED_FUSION_ALPHA
        )

    fixed_cal_acc = \
        accuracy_score(
            y_cal,
            P_fixed_cal.argmax(
                axis=1
            )
        )

    fixed_test_acc = \
        accuracy_score(
            y_test,
            P_fixed_test.argmax(
                axis=1
            )
        )

    # ========================================================
    # 11. CALIBRATION-SELECTED FUSION
    # ========================================================

    alpha_adaptive, \
        adaptive_cal_acc = \
        choose_calibration_alpha(
            P_csp_cal,
            P_cnn_cal,
            y_cal
        )

    P_adaptive_test = \
        confidence_fusion(
            P_csp_test,
            P_cnn_test,
            alpha_adaptive
        )

    adaptive_test_acc = \
        accuracy_score(
            y_test,
            P_adaptive_test.argmax(
                axis=1
            )
        )

    print(
        f"    Fixed fusion "
        f"alpha={FIXED_FUSION_ALPHA:.2f} "
        f"cal={fixed_cal_acc*100:.2f}% "
        f"test={fixed_test_acc*100:.2f}%"
    )

    print(
        f"    Adaptive fusion "
        f"alpha={alpha_adaptive:.2f} "
        f"cal={adaptive_cal_acc*100:.2f}% "
        f"test={adaptive_test_acc*100:.2f}%"
    )

    # ========================================================
    # 12. FINAL CANDIDATE SELECTION
    # ========================================================
    #
    # Restrict selection to:
    #   FBCSP
    #   fine-tuned CNN
    #   fixed fusion
    #
    # No dozens of candidates.
    # ========================================================

    candidates = {

        "FBCSP-LDA":
            (
                csp_cal_acc,
                P_csp_test
            ),

        "CNN":
            (
                cnn_cal_acc,
                P_cnn_test
            ),

        "Fixed-Fusion":
            (
                fixed_cal_acc,
                P_fixed_test
            )
    }

    best_method = max(
        candidates,
        key=lambda name:
            candidates[name][0]
    )

    best_cal_acc = \
        candidates[
            best_method
        ][0]

    best_probs = \
        candidates[
            best_method
        ][1]

    best_pred = \
        best_probs.argmax(
            axis=1
        )

    best_test_acc = \
        accuracy_score(
            y_test,
            best_pred
        )

    # ========================================================
    # 13. PRINT FINAL SUBJECT RESULT
    # ========================================================

    print(
        "\n"
        + "-" * 64
    )

    print(
        f"S{test_subject:03d} FINAL RESULT"
    )

    print(
        "-" * 64
    )

    print(
        f"FBCSP-LDA        : "
        f"{csp_test_acc*100:.2f}%"
    )

    print(
        f"Population CNN   : "
        f"{population_test_acc*100:.2f}%"
    )

    print(
        f"Fine-tuned CNN   : "
        f"{cnn_test_acc*100:.2f}%"
    )

    print(
        f"Fixed Fusion     : "
        f"{fixed_test_acc*100:.2f}%"
    )

    print(
        f"Adaptive Fusion  : "
        f"{adaptive_test_acc*100:.2f}%"
    )

    print(
        f"Selected         : "
        f"{best_method}"
    )

    print(
        f"FINAL TEST       : "
        f"{best_test_acc*100:.2f}%"
    )

    print(
        "-" * 64
    )

    # ========================================================
    # 14. SINGLE CONSISTENT RETURN STRUCTURE
    # ========================================================

    return {

        "subject":
            test_subject,

        "y_test":
            y_test,

        "best_method":
            best_method,

        "best_cal_acc":
            float(
                best_cal_acc
            ),

        "best_test_acc":
            float(
                best_test_acc
            ),

        "best_pred":
            best_pred,

        "best_probs":
            best_probs,

        "csp_test_acc":
            float(
                csp_test_acc
            ),

        "cnn_test_acc":
            float(
                cnn_test_acc
            ),

        "population_cnn_test_acc":
            float(
                population_test_acc
            ),

        "fixed_fusion_test_acc":
            float(
                fixed_test_acc
            ),

        "adaptive_fusion_test_acc":
            float(
                adaptive_test_acc
            ),

        "csp_cal_acc":
            float(
                csp_cal_acc
            ),

        "cnn_cal_acc":
            float(
                cnn_cal_acc
            ),

        "fixed_fusion_cal_acc":
            float(
                fixed_cal_acc
            ),

        "adaptive_fusion_cal_acc":
            float(
                adaptive_cal_acc
            ),

        "fixed_alpha":
            float(
                FIXED_FUSION_ALPHA
            ),

        "adaptive_alpha":
            float(
                alpha_adaptive
            ),

        "population_temperature":
            float(
                population_T
            ),

        "fine_tuned_temperature":
            float(
                fine_tuned_T
            ),

        "calibration_train_acc":
            float(
                calibration_train_acc
            ),

        "history":
            history,

        "model":
            model_finetuned
    }


print(
    "\n✅ CELL 15 READY."
)



✅ CELL 15 READY.


In [76]:

# CELL 16 — FINAL ERROR-PROOF LOSO RUNNER
# ============================================================
#
# Uses ONLY the keys returned by Cell 15.
#
# Saves checkpoint after every subject.
#
# ============================================================


def run_loso(
    subject_list,
    resume=False
):

    subject_list = [
        int(s)
        for s in subject_list
    ]

    rows = []

    fold_store = {}

    checkpoint_csv = (
        SAVE_DIR /
        "checkpoint_results.csv"
    )

    checkpoint_pkl = (
        SAVE_DIR /
        "checkpoint_folds.pkl"
    )

    # ========================================================
    # LOAD CHECKPOINT
    # ========================================================

    if (
        resume
        and
        checkpoint_csv.exists()
    ):

        print(
            "Loading checkpoint..."
        )

        old_df = pd.read_csv(
            checkpoint_csv
        )

        rows = old_df.to_dict(
            "records"
        )

        if checkpoint_pkl.exists():

            with open(
                checkpoint_pkl,
                "rb"
            ) as f:

                fold_store = pickle.load(
                    f
                )

        print(
            "Recovered:",
            len(rows),
            "subjects"
        )

    completed = {
        int(r["subject"])
        for r in rows
    }

    remaining = [
        s
        for s in subject_list
        if s not in completed
    ]

    print(
        "\n"
        + "=" * 72
    )

    print(
        "LOSO EXPERIMENT"
    )

    print(
        "=" * 72
    )

    print(
        "Requested subjects :",
        len(subject_list)
    )

    print(
        "Completed          :",
        len(completed)
    )

    print(
        "Remaining          :",
        len(remaining)
    )

    print(
        "=" * 72
    )

    total_start = time.time()

    # ========================================================
    # SUBJECT LOOP
    # ========================================================

    for i, sid in enumerate(
        remaining,
        1
    ):

        fold_start = time.time()

        print(
            "\n"
            + "#" * 72
        )

        print(
            f"SUBJECT "
            f"{i}/{len(remaining)} "
            f"— S{sid:03d}"
        )

        print(
            "#" * 72
        )

        try:

            fold = run_subject_fold(
                sid,
                verbose=(
                    len(subject_list)
                    <= 5
                )
            )

        except Exception as e:

            print(
                "\n"
                + "!" * 72
            )

            print(
                f"ERROR in S{sid:03d}"
            )

            print(
                repr(e)
            )

            print(
                "This subject is skipped."
            )

            print(
                "!" * 72
            )

            # Save completed subjects.
            pd.DataFrame(
                rows
            ).to_csv(
                checkpoint_csv,
                index=False
            )

            with open(
                checkpoint_pkl,
                "wb"
            ) as f:

                pickle.dump(
                    fold_store,
                    f
                )

            continue

        # ====================================================
        # SAVE COMPLETE FOLD
        # ====================================================

        fold_store[
            sid
        ] = fold

        # ====================================================
        # METRICS
        # ====================================================

        fold_minutes = (
            time.time()
            -
            fold_start
        ) / 60.0

        row = {

            "subject":
                sid,

            "best_method":
                fold[
                    "best_method"
                ],

            "best_cal_acc":
                fold[
                    "best_cal_acc"
                ] * 100.0,

            "best_test_acc":
                fold[
                    "best_test_acc"
                ] * 100.0,

            "csp_cal_acc":
                fold[
                    "csp_cal_acc"
                ] * 100.0,

            "csp_test_acc":
                fold[
                    "csp_test_acc"
                ] * 100.0,

            "population_cnn_test_acc":
                fold[
                    "population_cnn_test_acc"
                ] * 100.0,

            "cnn_cal_acc":
                fold[
                    "cnn_cal_acc"
                ] * 100.0,

            "cnn_test_acc":
                fold[
                    "cnn_test_acc"
                ] * 100.0,

            "fixed_fusion_cal_acc":
                fold[
                    "fixed_fusion_cal_acc"
                ] * 100.0,

            "fixed_fusion_test_acc":
                fold[
                    "fixed_fusion_test_acc"
                ] * 100.0,

            "adaptive_fusion_cal_acc":
                fold[
                    "adaptive_fusion_cal_acc"
                ] * 100.0,

            "adaptive_fusion_test_acc":
                fold[
                    "adaptive_fusion_test_acc"
                ] * 100.0,

            "fixed_alpha":
                fold[
                    "fixed_alpha"
                ],

            "adaptive_alpha":
                fold[
                    "adaptive_alpha"
                ],

            "population_temperature":
                fold[
                    "population_temperature"
                ],

            "fine_tuned_temperature":
                fold[
                    "fine_tuned_temperature"
                ],

            "calibration_train_acc":
                fold[
                    "calibration_train_acc"
                ] * 100.0,

            "fold_minutes":
                fold_minutes
        }

        rows.append(
            row
        )

        # ====================================================
        # RUNNING SUMMARY
        # ====================================================

        df_tmp = pd.DataFrame(
            rows
        )

        df_tmp = df_tmp.sort_values(
            "subject"
        )

        best_mean = df_tmp[
            "best_test_acc"
        ].mean()

        cnn_mean = df_tmp[
            "cnn_test_acc"
        ].mean()

        csp_mean = df_tmp[
            "csp_test_acc"
        ].mean()

        fusion_mean = df_tmp[
            "fixed_fusion_test_acc"
        ].mean()

        # ====================================================
        # PRINT
        # ====================================================

        print(
            "\n"
            + "-" * 72
        )

        print(
            f"S{sid:03d} COMPLETE"
        )

        print(
            "-" * 72
        )

        print(
            f"CSP                 : "
            f"{row['csp_test_acc']:.2f}%"
        )

        print(
            f"Population CNN      : "
            f"{row['population_cnn_test_acc']:.2f}%"
        )

        print(
            f"Fine-tuned CNN      : "
            f"{row['cnn_test_acc']:.2f}%"
        )

        print(
            f"Fixed Fusion        : "
            f"{row['fixed_fusion_test_acc']:.2f}%"
        )

        print(
            f"Selected             : "
            f"{row['best_method']}"
        )

        print(
            f"FINAL                : "
            f"{row['best_test_acc']:.2f}%"
        )

        print(
            "-" * 72
        )

        print(
            f"Running CSP mean    : "
            f"{csp_mean:.2f}%"
        )

        print(
            f"Running CNN mean    : "
            f"{cnn_mean:.2f}%"
        )

        print(
            f"Running fusion mean : "
            f"{fusion_mean:.2f}%"
        )

        print(
            f"Running best mean   : "
            f"{best_mean:.2f}%"
        )

        print(
            f"Fold time           : "
            f"{fold_minutes:.2f} min"
        )

        # ====================================================
        # CHECKPOINT
        # ====================================================

        df_tmp.to_csv(
            checkpoint_csv,
            index=False
        )

        with open(
            checkpoint_pkl,
            "wb"
        ) as f:

            pickle.dump(
                fold_store,
                f
            )

        print(
            "✅ Checkpoint saved."
        )

        # ====================================================
        # CLEANUP
        # ====================================================

        del fold

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ========================================================
    # FINAL DATAFRAME
    # ========================================================

    results_df = pd.DataFrame(
        rows
    )

    if len(results_df) == 0:

        print(
            "No successful subjects."
        )

        return (
            results_df,
            fold_store
        )

    results_df = results_df.sort_values(
        "subject"
    ).reset_index(
        drop=True
    )

    # ========================================================
    # FINAL STATISTICS
    # ========================================================

    best_mean = results_df[
        "best_test_acc"
    ].mean()

    best_std = results_df[
        "best_test_acc"
    ].std(
        ddof=0
    )

    best_median = results_df[
        "best_test_acc"
    ].median()

    csp_mean = results_df[
        "csp_test_acc"
    ].mean()

    cnn_mean = results_df[
        "cnn_test_acc"
    ].mean()

    fusion_mean = results_df[
        "fixed_fusion_test_acc"
    ].mean()

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 72
    )

    print(
        "FINAL LOSO RESULT"
    )

    print(
        "=" * 72
    )

    print(
        f"Subjects             : "
        f"{len(results_df)}"
    )

    print(
        f"FBCSP mean           : "
        f"{csp_mean:.2f}%"
    )

    print(
        f"CNN mean             : "
        f"{cnn_mean:.2f}%"
    )

    print(
        f"Fixed fusion mean    : "
        f"{fusion_mean:.2f}%"
    )

    print(
        f"Best candidate mean  : "
        f"{best_mean:.2f}%"
    )

    print(
        f"Best candidate std   : "
        f"{best_std:.2f}%"
    )

    print(
        f"Best candidate median: "
        f"{best_median:.2f}%"
    )

    print(
        f"75% target           : "
        f"{'✅ REACHED' if best_mean >= 75 else 'NOT REACHED'}"
    )

    print(
        f"Total time           : "
        f"{(time.time()-total_start)/3600:.2f} h"
    )

    print(
        "=" * 72
    )

    # ========================================================
    # SAVE FINAL
    # ========================================================

    final_csv = (
        SAVE_DIR /
        "module7_final_results.csv"
    )

    final_pkl = (
        SAVE_DIR /
        "module7_final_folds.pkl"
    )

    results_df.to_csv(
        final_csv,
        index=False
    )

    with open(
        final_pkl,
        "wb"
    ) as f:

        pickle.dump(
            fold_store,
            f
        )

    print(
        "\nSaved:"
    )

    print(
        final_csv
    )

    print(
        final_pkl
    )

    return (
        results_df,
        fold_store
    )


# ============================================================
# DEMO — FIRST FIVE
# ============================================================

demo_subjects = subjects_all[
    :DEMO_SUBJECTS
]

demo_results, demo_folds = \
    run_loso(
        demo_subjects,
        resume=False
    )

display(
    demo_results
)

if len(demo_results) > 0:

    print(
        "\n"
        + "=" * 72
    )

    print(
        "5-SUBJECT DEMO"
    )

    print(
        "=" * 72
    )

    print(
        "CSP mean:",
        f"{demo_results['csp_test_acc'].mean():.2f}%"
    )

    print(
        "CNN mean:",
        f"{demo_results['cnn_test_acc'].mean():.2f}%"
    )

    print(
        "Fusion mean:",
        f"{demo_results['fixed_fusion_test_acc'].mean():.2f}%"
    )

    print(
        "Best mean:",
        f"{demo_results['best_test_acc'].mean():.2f}%"
    )



LOSO EXPERIMENT
Requested subjects : 5
Completed          : 0
Remaining          : 5

########################################################################
SUBJECT 1/5 — S001
########################################################################

Preparing population EA for held-out S001...
Train: (19495, 32, 384) Cal: (90, 32, 384) Test: (90, 32, 384)
Cal classes: [32 28 14 16]
Test classes: [14 16 28 32]

Training FBCSP-LDA...
    FBCSP bands: [(6, 10), (8, 12), (10, 14), (12, 18), (16, 24), (20, 30), (24, 36)]
      Band 1/7: 6-10 Hz ...features=24
      Band 2/7: 8-12 Hz ...features=24
      Band 3/7: 10-14 Hz ...features=24
      Band 4/7: 12-18 Hz ...features=24
      Band 5/7: 16-24 Hz ...features=24
      Band 6/7: 20-30 Hz ...features=24
      Band 7/7: 24-36 Hz ...features=24

    Raw FBCSP feature shapes:
      Train: (19495, 168)
      Cal  : (90, 168)
      Test : (90, 168)

    FBCSP complete.
      Final features: 168
      Classes: [0 1 2 3]
    CSP calibration: 4

KeyboardInterrupt: 

In [ ]:
# CELL 17 — Results visualization
# ============================================================

def plot_results(
    results_df
):

    df = results_df.sort_values(
        "subject"
    )

    # --------------------------------------------------------
    # Subject-wise best accuracy
    # --------------------------------------------------------

    plt.figure(
        figsize=(16, 5)
    )

    plt.bar(
        df["subject"].astype(str),
        df["best_test_acc"]
    )

    plt.axhline(
        25,
        linestyle=":",
        linewidth=1.5,
        label="Chance"
    )

    plt.axhline(
        75,
        linestyle="--",
        linewidth=1.5,
        label="Target 75%"
    )

    plt.xlabel(
        "Held-out Subject"
    )

    plt.ylabel(
        "Accuracy (%)"
    )

    plt.title(
        "Module 7-P — Subject-Independent LOSO Accuracy"
    )

    plt.legend()

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.xticks(
        rotation=90,
        fontsize=7
    )

    plt.tight_layout()

    plt.show()

    # --------------------------------------------------------
    # Branch comparison
    # --------------------------------------------------------

    methods = [
        "FBCSP-LDA",
        "CNN",
        "Adaptive Fusion",
        "Fixed Fusion",
        "Best Candidate"
    ]

    values = [
        df["csp_test_acc"].mean(),
        df["cnn_test_acc"].mean(),
        df["adaptive_fusion_test_acc"].mean(),
        df["fixed_fusion_test_acc"].mean(),
        df["best_test_acc"].mean()
    ]

    plt.figure(
        figsize=(10, 5)
    )

    plt.bar(
        methods,
        values
    )

    plt.axhline(
        75,
        linestyle="--",
        linewidth=1.5,
        label="Target 75%"
    )

    plt.ylabel(
        "Mean LOSO Accuracy (%)"
    )

    plt.title(
        "Module 7-P Branch Comparison"
    )

    plt.legend()

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.xticks(
        rotation=15
    )

    plt.tight_layout()

    plt.show()


def plot_confusion_from_folds(
    fold_store
):

    y_all = []
    p_all = []

    for sid in sorted(
        fold_store.keys()
    ):

        fold = fold_store[
            sid
        ]

        y_all.extend(
            fold["y_test"]
        )

        p_all.extend(
            fold["best_pred"]
        )

    y_all = np.asarray(
        y_all
    )

    p_all = np.asarray(
        p_all
    )

    cm = confusion_matrix(
        y_all,
        p_all,
        labels=list(
            range(N_CLASSES)
        )
    )

    row_sum = \
        cm.sum(
            axis=1,
            keepdims=True
        )

    cm_pct = (
        cm /
        np.maximum(
            row_sum,
            1
        )
    ) * 100.0

    plt.figure(
        figsize=(7, 6)
    )

    plt.imshow(
        cm_pct,
        aspect="auto"
    )

    plt.colorbar(
        label="Recall (%)"
    )

    plt.xticks(
        range(N_CLASSES),
        CLASS_NAMES,
        rotation=30,
        ha="right"
    )

    plt.yticks(
        range(N_CLASSES),
        CLASS_NAMES
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        "Pooled LOSO Confusion Matrix"
    )

    for i in range(
        N_CLASSES
    ):

        for j in range(
            N_CLASSES
        ):

            plt.text(
                j,
                i,
                f"{cm_pct[i,j]:.1f}",
                ha="center",
                va="center"
            )

    plt.tight_layout()

    plt.show()


print(
    "Visualization functions ready."
)

In [ ]:
# CELL 18 — Final classification report
# ============================================================

def final_report(
    results_df,
    fold_store
):

    y_all = []
    p_all = []

    for sid in sorted(
        fold_store.keys()
    ):

        fold = fold_store[
            sid
        ]

        y_all.extend(
            fold["y_test"]
        )

        p_all.extend(
            fold["best_pred"]
        )

    y_all = np.asarray(
        y_all
    )

    p_all = np.asarray(
        p_all
    )

    print(
        classification_report(
            y_all,
            p_all,
            labels=list(
                range(N_CLASSES)
            ),
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0
        )
    )

    balanced = \
        balanced_accuracy_score(
            y_all,
            p_all
        )

    print(
        f"Pooled balanced accuracy: "
        f"{balanced*100:.2f}%"
    )

    # --------------------------------------------------------
    # Per-subject class recall
    # --------------------------------------------------------

    rows = []

    for sid in sorted(
        fold_store.keys()
    ):

        yt = np.asarray(
            fold_store[
                sid
            ]["y_test"]
        )

        yp = np.asarray(
            fold_store[
                sid
            ]["best_pred"]
        )

        row = {
            "subject": sid
        }

        for cls, name in enumerate(
            CLASS_NAMES
        ):

            mask = yt == cls

            if mask.any():

                row[name] = (
                    yp[mask] == cls
                ).mean() * 100.0

            else:

                row[name] = np.nan

        rows.append(
            row
        )

    per_class_df = pd.DataFrame(
        rows
    )

    print(
        "\nMean subject-wise class recall:"
    )

    display(
        per_class_df.mean(
            numeric_only=True
        ).to_frame(
            "Mean Recall (%)"
        )
    )

    per_class_df.to_csv(
        SAVE_DIR /
        "module7_per_class_recall.csv",
        index=False
    )

    return per_class_df


print(
    "Final report function ready."
)

In [ ]:
# CELL 19 — Final summary + run after LOSO
# ============================================================

def make_project_summary(
    results_df
):

    summary = {

        "dataset":
            "PhysioNet EEGMMIDB",

        "subjects_evaluated":
            int(
                len(results_df)
            ),

        "task":
            "4-class motor imagery",

        "classes":
            CLASS_NAMES,

        "target_runs":
            TARGET_RUNS,

        "calibration_runs":
            CALIBRATION_RUNS,

        "test_runs":
            TEST_RUNS,

        "sampling_rate":
            TARGET_FS,

        "bandpass":
            [
                BP_LO,
                BP_HI
            ],

        "epoch_window":
            [
                EPOCH_TMIN,
                EPOCH_TMAX
            ],

        "second_window":
            (
                [
                    SECOND_TMIN,
                    SECOND_TMAX
                ]
                if USE_SECOND_WINDOW
                else None
            ),

        "channels":
            channels,

        "architecture":
            [
                "Calibration-only Euclidean Alignment",
                "FBCSP + shrinkage LDA",
                "Multi-scale temporal CNN",
                "Depthwise spatial convolution",
                "Dilated residual temporal blocks",
                "SE attention",
                "Center Loss",
                "Label smoothing",
                "Subject calibration fine-tuning",
                "Temperature scaling",
                "Conservative probability fusion"
            ],

        "mean_best_accuracy":
            float(
                results_df[
                    "best_test_acc"
                ].mean()
            ),

        "std_best_accuracy":
            float(
                results_df[
                    "best_test_acc"
                ].std(
                    ddof=0
                )
            ),

        "median_best_accuracy":
            float(
                results_df[
                    "best_test_acc"
                ].median()
            ),

        "mean_csp":
            float(
                results_df[
                    "csp_test_acc"
                ].mean()
            ),

        "mean_cnn":
            float(
                results_df[
                    "cnn_test_acc"
                ].mean()
            ),

        "mean_fixed_fusion":
            float(
                results_df[
                    "fixed_fusion_test_acc"
                ].mean()
            )
    }

    summary_path = (
        SAVE_DIR /
        "module7_project_summary.json"
    )

    with open(
        summary_path,
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )

    print(
        json.dumps(
            summary,
            indent=2
        )
    )

    print(
        "\nSaved:",
        summary_path
    )

    return summary

## Recommended submission wording

This is a **calibration-assisted subject-independent** experiment: the population model is trained on the other subjects; calibration runs from the held-out subject are used only for adaptation and calibration; the held-out test runs are untouched.

Do not state “75% achieved” until Cell 16 actually reports ≥75% mean LOSO accuracy. If your score is below 75%, report the measured mean ± standard deviation and identify the final configuration honestly.

For your deadline, get the non-GAN version running first. Only turn on `USE_FBGAN=True` after the baseline/fusion pipeline is verified; it is an optional ablation rather than a requirement for the core method.
